# Combine GEE Features and PM25 Data

This notebook merges two CSV files:
- **gee_features_daily.csv**: Environmental features (temperature, pressure, wind, NO2, CO, O3, AOD)
- **pm25.csv**: PM2.5 air quality measurements and sensor locations

The merge is performed on three key fields:
- `date`: Calendar date
- `latitude`: Geographic latitude coordinate
- `longitude`: Geographic longitude coordinate

Output: **combined_gee_pm25.csv** containing all features and PM2.5 levels for each location on each date.

## Step 0: Import Libraries and Setup Paths

In [10]:
import pandas as pd
from pathlib import Path

# Get the base directory (workspace root)
notebook_dir = Path.cwd()
# If running from scripts folder, go up one level
if notebook_dir.name == 'notebooks':
    base_dir = notebook_dir.parent
else:
    base_dir = notebook_dir

data_dir = base_dir / 'data'

print(f"Base directory: {base_dir}")
print(f"Data directory: {data_dir}")
print(f"\nData files present:")
for file in data_dir.glob('*.csv'):
    print(f"  - {file.name}")

Base directory: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality\Global_approach
Data directory: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality\Global_approach\data

Data files present:
  - gee_features_daily.csv
  - GlobalWeatherRepository.csv
  - land_features.csv


## Step 1: Load Data Files

In [3]:
print("Loading data files...\n")

# Load the CSV files
pm25_df = pd.read_csv(data_dir / 'GlobalWeatherRepository.csv')
gee_df = pd.read_csv(data_dir / 'gee_features_daily.csv')

print(f"PM25 data loaded: {pm25_df.shape[0]} rows × {pm25_df.shape[1]} columns")
print(f"GEE data loaded: {gee_df.shape[0]} rows × {gee_df.shape[1]} columns")

print(f"\nPM25 columns: {list(pm25_df.columns)}")
print(f"\nGEE columns: {list(gee_df.columns)}")

Loading data files...

PM25 data loaded: 139168 rows × 41 columns
GEE data loaded: 186013 rows × 5 columns

PM25 columns: ['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'last_updated', 'temperature_celsius', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'air_quality_PM2.5', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination']

GEE columns: ['date', 'id', 'latitude', 'longitude', 'AOD']


## Step 2: Preview Data

In [4]:
print("First few rows of PM25 data:")
print(pm25_df.head())


print("\nFirst few rows of GEE data:")
print(gee_df.head())

First few rows of PM25 data:
       country     location_name  latitude  longitude        timezone  \
0  Afghanistan             Kabul     34.52      69.18      Asia/Kabul   
1      Albania            Tirana     41.33      19.82   Europe/Tirane   
2      Algeria           Algiers     36.76       3.05  Africa/Algiers   
3      Andorra  Andorra La Vella     42.50       1.52  Europe/Andorra   
4       Angola            Luanda     -8.84      13.23   Africa/Luanda   

   last_updated_epoch      last_updated  temperature_celsius  \
0          1715849100  2024-05-16 13:15                 26.6   
1          1715849100  2024-05-16 10:45                 19.0   
2          1715849100  2024-05-16 09:45                 23.0   
3          1715849100  2024-05-16 10:45                  6.3   
4          1715849100  2024-05-16 09:45                 26.0   

   temperature_fahrenheit condition_text  ...  air_quality_PM2.5  \
0                    79.8  Partly Cloudy  ...                8.4   
1          

In [5]:
print(pm25_df.shape)
print(gee_df.shape)

(139168, 41)
(186013, 5)


In [6]:
pm25_df['air_quality_PM2.5']

0           8.40
1           1.10
2          10.40
3           0.70
4         183.40
           ...  
139163     13.75
139164     66.55
139165      8.85
139166      7.55
139167     16.45
Name: air_quality_PM2.5, Length: 139168, dtype: float64

## Step 3: Standardize Column Names

In [7]:
print("Standardizing column names...")
print(f"Original GEE columns: {list(gee_df.columns)}")

# Rename lat/lon to latitude/longitude in GEE data for consistency
gee_df = gee_df.rename(columns={'lat': 'latitude', 'lon': 'longitude'})
pm25_df = pm25_df.rename(columns={'lat': 'latitude', 'lon': 'longitude' ,'air_quality_PM2.5' : 'pm25' , 'last_updated' : 'date'})

print(f"After rename: {list(gee_df.columns)}")
print(f"After rename: {list(pm25_df.columns)}")

Standardizing column names...
Original GEE columns: ['date', 'id', 'latitude', 'longitude', 'AOD']
After rename: ['date', 'id', 'latitude', 'longitude', 'AOD']
After rename: ['country', 'location_name', 'latitude', 'longitude', 'timezone', 'last_updated_epoch', 'date', 'temperature_celsius', 'temperature_fahrenheit', 'condition_text', 'wind_mph', 'wind_kph', 'wind_degree', 'wind_direction', 'pressure_mb', 'pressure_in', 'precip_mm', 'precip_in', 'humidity', 'cloud', 'feels_like_celsius', 'feels_like_fahrenheit', 'visibility_km', 'visibility_miles', 'uv_index', 'gust_mph', 'gust_kph', 'air_quality_Carbon_Monoxide', 'air_quality_Ozone', 'air_quality_Nitrogen_dioxide', 'air_quality_Sulphur_dioxide', 'pm25', 'air_quality_PM10', 'air_quality_us-epa-index', 'air_quality_gb-defra-index', 'sunrise', 'sunset', 'moonrise', 'moonset', 'moon_phase', 'moon_illumination']


## Step 4: Convert Date Columns to Datetime Format

In [14]:
print("Converting date columns to datetime format...\n")

pm25_df['date'] = pd.to_datetime(pm25_df['date'], format='mixed', errors='coerce').dt.normalize()
gee_df['date'] = pd.to_datetime(gee_df['date'], format='mixed', errors='coerce').dt.normalize()

if pm25_df['date'].isna().any():
    print(f"Warning: {pm25_df['date'].isna().sum()} PM25 rows could not be parsed as dates.")
if gee_df['date'].isna().any():
    print(f"Warning: {gee_df['date'].isna().sum()} GEE rows could not be parsed as dates.")

print(f"PM25 date range: {pm25_df['date'].min().date()} to {pm25_df['date'].max().date()}")
print(f"GEE date range: {gee_df['date'].min().date()} to {gee_df['date'].max().date()}")

print(f"\nPM25: {pm25_df['date'].nunique()} unique dates")
print(f"GEE: {gee_df['date'].nunique()} unique dates")

Converting date columns to datetime format...

PM25 date range: 2024-05-16 to 2026-05-03
GEE date range: 2024-05-16 to 2026-05-01

PM25: 717 unique dates
GEE: 715 unique dates


## Step 5: Merge Datasets on Date, Latitude, and Longitude

In [15]:
print("Merging datasets on date, latitude, and longitude...\n")
print(f"Merge type: Inner join (only matching rows in both datasets)")
print(f"Key fields: date, latitude, longitude\n")

merged_df = pd.merge(
    gee_df,
    pm25_df,
    on=['date', 'latitude', 'longitude'],
    how='inner'
)

print(f"✓ Merge completed successfully!")
print(f"Merged data shape: {merged_df.shape[0]} rows × {merged_df.shape[1]} columns")
print(f"\nColumns in merged dataset ({merged_df.shape[1]}):")
for i, col in enumerate(merged_df.columns, 1):
    print(f"  {i:2d}. {col}")

Merging datasets on date, latitude, and longitude...

Merge type: Inner join (only matching rows in both datasets)
Key fields: date, latitude, longitude

✓ Merge completed successfully!
Merged data shape: 83704 rows × 43 columns

Columns in merged dataset (43):
   1. date
   2. id
   3. latitude
   4. longitude
   5. AOD
   6. country
   7. location_name
   8. timezone
   9. last_updated_epoch
  10. temperature_celsius
  11. temperature_fahrenheit
  12. condition_text
  13. wind_mph
  14. wind_kph
  15. wind_degree
  16. wind_direction
  17. pressure_mb
  18. pressure_in
  19. precip_mm
  20. precip_in
  21. humidity
  22. cloud
  23. feels_like_celsius
  24. feels_like_fahrenheit
  25. visibility_km
  26. visibility_miles
  27. uv_index
  28. gust_mph
  29. gust_kph
  30. air_quality_Carbon_Monoxide
  31. air_quality_Ozone
  32. air_quality_Nitrogen_dioxide
  33. air_quality_Sulphur_dioxide
  34. pm25
  35. air_quality_PM10
  36. air_quality_us-epa-index
  37. air_quality_gb-defra-ind

## Step 6: Sort Data for Readability

In [16]:
print("Sorting data by date and coordinates...\n")

merged_df = merged_df.sort_values(['date', 'latitude', 'longitude']).reset_index(drop=True)

print("✓ Data sorted by date, latitude, longitude")
print(f"\nFirst few rows of merged data:")
print(merged_df.head())

Sorting data by date and coordinates...

✓ Data sorted by date, latitude, longitude

First few rows of merged data:
        date  id  latitude  longitude       AOD      country location_name  \
0 2024-05-16   0    -41.30     174.78  0.149667  New Zealand    Wellington   
1 2024-05-16   4    -35.28     149.22  0.120500    Australia      Canberra   
2 2024-05-16   7    -34.59     -58.67  0.020000    Argentina  Buenos Aires   
3 2024-05-16   7    -34.59     -58.67  0.020000    Argentina  Buenos Aires   
4 2024-05-16   9    -33.45     -70.67  0.178500        Chile      Santiago   

                         timezone  last_updated_epoch  temperature_celsius  \
0                Pacific/Auckland          1715849100                 13.0   
1                Australia/Sydney          1715849100                  9.0   
2  America/Argentina/Buenos_Aires          1715849100                  8.0   
3  America/Argentina/Buenos_Aires          1715868000                 11.0   
4                America/

## Step 7: Save Combined Dataset

In [17]:
print(data_dir)


c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality\Global_approach\data


In [18]:
print("Saving combined dataset...\n")

output_path = data_dir / 'combined_gee_pm25.csv'
merged_df.to_csv(output_path, index=False)

print(f"✓ Output saved to: {output_path}")
print(f"\nFile size: {output_path.stat().st_size / 1024:.2f} KB")

Saving combined dataset...

✓ Output saved to: c:\Users\sarah\Desktop\Groupe Project\pm25-air-quality\Global_approach\data\combined_gee_pm25.csv

File size: 21928.52 KB


## Summary and Verification

In [19]:
print("="*70)
print("MERGE SUMMARY")
print("="*70)

print(f"\nInput files:")
print(f"  PM25:                {pm25_df.shape[0]:,} rows")
print(f"  GEE Features:        {gee_df.shape[0]:,} rows")

print(f"\nOutput file (combined_gee_pm25.csv):")
print(f"  Rows:                {merged_df.shape[0]:,}")
print(f"  Columns:             {merged_df.shape[1]}")

print(f"\nData coverage:")
print(f"  Date range:          {merged_df['date'].min().date()} to {merged_df['date'].max().date()}")
print(f"  Unique dates:        {merged_df['date'].nunique()}")
print(f"  Unique locations:    {len(merged_df[['latitude', 'longitude']].drop_duplicates())}")

print(f"\nMissing values per column:")
missing = merged_df.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) == 0:
    print("  ✓ No missing values!")
else:
    for col, count in missing_cols.items():
        pct = (count / len(merged_df)) * 100
        print(f"  {col}: {count:,} ({pct:.1f}%)")

print("\n" + "="*70)
print("✓ Merge completed successfully!")
print("="*70)

MERGE SUMMARY

Input files:
  PM25:                139,168 rows
  GEE Features:        186,013 rows

Output file (combined_gee_pm25.csv):
  Rows:                83,704
  Columns:             43

Data coverage:
  Date range:          2024-05-16 to 2026-05-01
  Unique dates:        715
  Unique locations:    423

Missing values per column:
  ✓ No missing values!

✓ Merge completed successfully!


## Sample Data Inspection

In [20]:
print("Sample rows from combined dataset:\n")
print(merged_df.sample(min(5, len(merged_df))).to_string())

print("\n\nData info:")
print(merged_df.info())

Sample rows from combined dataset:

            date   id  latitude  longitude     AOD      country      location_name         timezone  last_updated_epoch  temperature_celsius  temperature_fahrenheit condition_text  wind_mph  wind_kph  wind_degree wind_direction  pressure_mb  pressure_in  precip_mm  precip_in  humidity  cloud  feels_like_celsius  feels_like_fahrenheit  visibility_km  visibility_miles  uv_index  gust_mph  gust_kph  air_quality_Carbon_Monoxide  air_quality_Ozone  air_quality_Nitrogen_dioxide  air_quality_Sulphur_dioxide     pm25  air_quality_PM10  air_quality_us-epa-index  air_quality_gb-defra-index   sunrise    sunset  moonrise   moonset       moon_phase  moon_illumination
20582 2024-11-03  404   51.2500     3.6333  0.2490      Belgium  'S Gravenjansdijk  Europe/Brussels          1730625300                  9.2                    48.6          Sunny       7.6      12.2           91              E       1030.0        30.41       0.00        0.0        80      2         

In [21]:
# unique lat and log 
df_unique_locations = merged_df[['latitude', 'longitude']].drop_duplicates()
print(f"\nUnique locations (latitude, longitude): {len(df_unique_locations)}")


Unique locations (latitude, longitude): 423
